# Week 31, Day 1: Special methods: a tour

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w031d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Shelf:
    def __init__(self, books):
        self.books = list(books)

    def __len__(self):
        return len(self.books)

full = Shelf(["Aké", "Small Island"])
empty = Shelf([])
print(len(full), len(empty))
print(bool(full), bool(empty), "yes" if empty else "no")

In [ ]:
class Shelf:
    def __init__(self, books):
        self.books = list(books)

s = Shelf(["Aké"])
s.__len__ = lambda: 1             # an attribute on this object only
print(s.__len__())
try:
    len(s)
except TypeError as err:
    print("TypeError:", err)

In [ ]:
class Queue:
    def __init__(self):
        self._people = []

    def join(self, name):
        self._people.append(name)

    def serve(self):
        return self._people.pop(0)

    def __len__(self):
        return len(self._people)

q = Queue()
q.join("Ada")
q.join("Bo")
while q:                          # true while len(q) isn't 0
    print("serving", q.serve(), "with", len(q), "left")
print(bool(q), len(q))

In [ ]:
class Sheets:
    def __init__(self, pages):
        self.pages = pages

    def __len__(self):
        return self.pages / 2          # a float, even for 10 pages

try:
    len(Sheets(10))
except TypeError as err:
    print("TypeError:", err)

class Wallet:
    def __init__(self, pence):
        self.pence = pence

    def __bool__(self):
        return self.pence              # an int, not a bool

try:
    bool(Wallet(5))
except TypeError as err:
    print("TypeError:", err)

In [ ]:
class Money:
    def __init__(self, pence):
        self.pence = pence

    def __int__(self):
        return self.pence

    def __float__(self):
        return self.pence / 100

    def __round__(self, ndigits=None):
        pounds = round(self.pence / 100, ndigits)
        return pounds

price = Money(1249)
print(int(price), float(price), round(price), round(price, 1))

In [ ]:
class Floor:
    def __init__(self, number):
        self.number = number

    def __index__(self):
        return self.number

    def __repr__(self):
        return f"Floor({self.number})"

names = ["ground", "first", "second", "third"]
f = Floor(2)
print(names[f], list(range(f)), bin(Floor(5)), hex(Floor(255)))
print(int(f))      # int() also uses __index__ when there's no __int__

In [ ]:
class Distance:
    def __init__(self, metres):
        self.metres = metres

    def __str__(self):
        return f"{self.metres} m"

    def __format__(self, spec):
        if spec == "":
            return str(self)
        if spec == "km":
            return f"{self.metres / 1000:.1f} km"
        if spec == "miles":
            return f"{self.metres / 1609.344:.1f} miles"
        return format(self.metres, spec)       # hand anything else to the number

d = Distance(42195)
print(f"{d} | {d:km} | {d:miles} | {d:,} | {d:>10}")
try:
    f"{d:parsecs}"
except ValueError as err:
    print("ValueError:", err)

In [ ]:
class Logged:
    """Wraps any object, and logs which attributes are used through it."""

    def __init__(self, inner):
        self.inner = inner
        self.used = []

    def __getattr__(self, name):           # only for names Logged doesn't have
        self.used.append(name)
        return getattr(self.inner, name)

shopping = Logged([])
shopping.append("yam")
shopping.append("eggs")
shopping.extend(["rice", "pepper"])
print(shopping.inner, shopping.count("rice"))
print(shopping.used)

> **Key idea.** Special methods connect your classes to Python's syntax and built-ins; you define them and Python calls them, always looking them up on the class. `__len__` gives a size (an `int` of 0 or more) and, without `__bool__`, decides truth; `__bool__` must return a real `bool`. `__int__`, `__float__` and `__round__` give conversions when there's one obvious answer; `__index__` says the object is a whole number, usable as a list position or in `range`. `__format__` gives meaning to format specs, handing unknown ones on to a value inside. `__getattr__` runs only when normal lookup fails, which makes it the tool for wrappers that forward to another object.

### Exercise: A playlist with a length

Give `Playlist` a `__len__` (the number of songs) and nothing else, so that `len` works and an empty playlist is false. Then write `longest(playlists)`, returning the playlist with the most songs (the first of any tie), or `None` if there are no playlists.

In [ ]:
class Playlist:
    def __init__(self, name, songs=()):
        self.name = name
        self.songs = list(songs)

    ___

def longest(playlists):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_len():\n    \"length and truth\"\n    P = ns[\"Playlist\"]\n    a, b, c = P(\"Road trip\", [\"Ye\", \"Essence\", \"Calm Down\"]), P(\"Empty\"), P(\"Two\", [\"a\", \"b\"])\n    assert len(a) == 3 and len(b) == 0 and bool(a) and not b\n    assert \"__bool__\" not in vars(P), \"truth should come from __len__\"\ndef test_longest():\n    \"the longest\"\n    P = ns[\"Playlist\"]\n    a, b, c = P(\"A\", [\"x\", \"y\"]), P(\"B\", [\"x\", \"y\"]), P(\"C\", [\"x\"])\n    assert longest([c, a, b]) is a and longest([]) is None and longest([P(\"E\")]).name == \"E\"")

### Exercise: Money conversions

Give `Money` these conversions: `int(m)` is the amount in pence, `float(m)` is the amount in pounds (`Money(1250)` gives `12.5`), and `bool(m)` is True unless the amount is 0. `__bool__` must return a real `bool`.

In [ ]:
class Money:
    def __init__(self, pence):
        self.pence = pence

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"conversions\"\n    M = ns[\"Money\"]\n    assert int(M(1250)) == 1250 and float(M(1250)) == 12.5 and float(M(5)) == 0.05\n    assert bool(M(1)) is True and bool(M(0)) is False and bool(M(-300)) is True\n    assert isinstance(float(M(100)), float) and isinstance(int(M(3)), int)\n    assert [\"no\", \"yes\"][bool(M(0))] == \"no\"")

### Exercise: Seat numbers as indexes

`Seat(row, number)` is a seat in a small theatre with 10 seats per row, rows `"A"` to `"J"`. Give it an `__index__` returning its position counting from 0 (`A1` is 0, `A10` is 9, `B1` is 10, `J10` is 99), so a seat can be used directly as a list index and in `range`. A row outside `A` to `J` or a number outside 1 to 10 raises `ValueError`. `repr` gives `Seat('B', 1)`.

In [ ]:
class Seat:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_index():\n    \"positions\"\n    S = ns[\"Seat\"]\n    assert [S(\"A\", 1).__index__(), S(\"A\", 10).__index__(), S(\"B\", 1).__index__(), S(\"J\", 10).__index__()] == [0, 9, 10, 99]\n    booked = [False] * 100\n    booked[S(\"C\", 4)] = True\n    assert booked.index(True) == 23 and len(range(S(\"A\", 5))) == 4 and repr(S(\"B\", 1)) == \"Seat('B', 1)\"\n    for bad in [(\"K\", 1), (\"A\", 0), (\"A\", 11), (\"a\", 1)]:\n        try:\n            S(*bad)\n            assert False, f\"Seat{bad} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Formatting durations

Write `Duration(seconds)` (a whole number of 0 or more, otherwise `ValueError`) with `__format__` supporting these specs:

- `""`: hours, minutes and seconds as `1:02:05` (minutes and seconds always two digits; hours as many as needed);
- `"m"`: whole minutes, rounded down, as `62m`;
- `"s"`: the seconds, as `3725s`;
- anything else raises `ValueError`.

Also give it `__int__` (the seconds) and make an empty duration (0 seconds) false.

In [ ]:
class Duration:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_format():\n    \"formats\"\n    D = ns[\"Duration\"]\n    d = D(3725)\n    assert f\"{d}\" == \"1:02:05\" and f\"{d:m}\" == \"62m\" and f\"{d:s}\" == \"3725s\", f\"got {d}, {d:m}, {d:s}\"\n    assert f\"{D(0)}\" == \"0:00:00\" and f\"{D(59):m}\" == \"0m\" and f\"{D(90061)}\" == \"25:01:01\"\n    assert format(D(60), \"m\") == \"1m\" and int(D(42)) == 42\n    assert not D(0) and D(1)\n    try:\n        f\"{d:hours}\"\n        assert False, \"an unknown spec should raise ValueError\"\n    except ValueError:\n        pass\n    for bad in [-1, 2.5]:\n        try:\n            D(bad)\n            assert False, f\"Duration({bad}) should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: A read-only view

Write `ReadOnly(inner)`, a wrapper that lets code **read** anything from the wrapped object but not change it through the wrapper. Forward attribute lookups with `__getattr__`, except that names in the class attribute `BLOCKED` (`append`, `extend`, `insert`, `remove`, `pop`, `clear`, `sort`, `reverse`, `update`, `setdefault`, `popitem`, `add`, `discard`) raise `AttributeError` with the message `read-only: append` (using the real name). Also give it `__len__` (the wrapped object's length), since special methods aren't forwarded.

In [ ]:
class ReadOnly:
    BLOCKED = {"append", "extend", "insert", "remove", "pop", "clear", "sort",
               "reverse", "update", "setdefault", "popitem", "add", "discard"}
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_list():\n    \"a list\"\n    R = ns[\"ReadOnly\"]\n    data = [3, 1, 2, 3]\n    r = R(data)\n    assert r.count(3) == 2 and r.index(2) == 2 and len(r) == 4\n    for name in [\"append\", \"sort\", \"pop\"]:\n        try:\n            getattr(r, name)\n            assert False, f\"{name} should be blocked\"\n        except AttributeError as err:\n            assert str(err) == f\"read-only: {name}\", f\"message was {str(err)!r}\"\n    assert data == [3, 1, 2, 3]\ndef test_dict():\n    \"a dictionary\"\n    R = ns[\"ReadOnly\"]\n    r = R({\"Lagos\": 15, \"Kano\": 4})\n    assert r.get(\"Kano\") == 4 and sorted(r.keys()) == [\"Kano\", \"Lagos\"] and len(r) == 2\n    try:\n        r.update({\"x\": 1})\n        assert False, \"update should be blocked\"\n    except AttributeError:\n        pass\n    try:\n        r.nonsense\n        assert False, \"a missing attribute should still raise AttributeError\"\n    except AttributeError:\n        pass")

### Bug hunt: Not a real bool

Checking whether a basket has anything in it crashes with `TypeError: __bool__ should return bool, returned int`. Fix it so it prints `has items` and then `empty`.

In [ ]:
class Basket:
    def __init__(self, items):
        self.items = list(items)

    def __bool__(self):
        return len(self.items)

for b in [Basket(["bread"]), Basket([])]:
    print("has items" if b else "empty")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both baskets\"\n    expect_output(\"has items\\nempty\")\n    assert bool(ns[\"Basket\"]([1])) is True")

### Bug hunt: Half a length

A booklet's length should be the number of sheets of paper (two pages per sheet, rounded up). It crashes with `TypeError: 'float' object cannot be interpreted as an integer`. Fix it so it prints `[5, 6, 1]`.

In [ ]:
class Booklet:
    def __init__(self, pages):
        self.pages = pages

    def __len__(self):
        return self.pages / 2

print([len(Booklet(p)) for p in [10, 11, 1]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sheets\"\n    expect_output(\"[5, 6, 1]\")\n    assert len(ns[\"Booklet\"](0)) == 0")

### Bug hunt: The repr that never shows

`Ticket` sets up its own repr in `__init__`, but printing a list of tickets still shows `<__main__.Ticket object at 0x...>`. Fix it so it prints `[Ticket('Lagos', 'Abuja')]`.

In [ ]:
class Ticket:
    def __init__(self, origin, destination):
        self.origin = origin
        self.destination = destination
        self.__repr__ = lambda: f"Ticket({self.origin!r}, {self.destination!r})"

print([Ticket("Lagos", "Abuja")])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the repr\"\n    expect_output(\"[Ticket('Lagos', 'Abuja')]\")\n    assert \"__repr__\" in vars(ns[\"Ticket\"]), \"special methods belong in the class body\"")

### Puzzle: A deck with a size

Arrange the lines so that a `Deck` has a size, and so is false when it's empty. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class A:
    def __len__(self):
        return 0

class B:
    def __len__(self):
        return 3
    def __bool__(self):
        return False

class C:
    pass

class D:
    def __index__(self):
        return 2

p = ___   # bool(A())
q = ___   # bool(B())
r = ___   # len(B())
s = ___   # bool(C())
t = ___   # "abcde"[D()]
u = ___   # int(D())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r, s) == (False, False, 3, True), \"p to s\"\n    assert (t, u) == (\"c\", 2)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: ratios

Write `Ratio(n, d)`, a fraction of two whole numbers kept in lowest terms, with the sign in the numerator: `Ratio(6, -8)` is stored as `n = -3`, `d = 4`. A `d` of 0 raises `ValueError`.

- `str` gives `-3/4`; `repr` gives `Ratio(-3, 4)`.
- `float(r)` is `n / d`. `int(r)` drops the fraction part, towards zero, like `int(-0.75)`, which is 0. `round(r)` and `round(r, k)` are `round(n / d)` and `round(n / d, k)`.
- `bool(r)` is False only for zero.
- `__index__` works only for whole numbers (`d == 1`), returning `n`; otherwise it raises `TypeError`. So `"abc"[Ratio(4, 2)]` is `"c"`.
- `__format__`: an empty spec gives `str(r)`; `"mixed"` gives a mixed number (`Ratio(7, 4)` is `1 3/4`, `Ratio(-7, 4)` is `-1 3/4`, `Ratio(8, 4)` is `2`, `Ratio(3, 4)` is `3/4`, `Ratio(0, 5)` is `0`); any other spec is handed to `format(float(r), spec)`, so `f"{Ratio(1, 3):.2f}"` is `0.33` and `f"{Ratio(3, 4):.0%}"` is `75%`.

In [ ]:
import math

class Ratio:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basics():\n    \"storage and text\"\n    R = ns[\"Ratio\"]\n    r = R(6, -8)\n    assert (r.n, r.d) == (-3, 4) and str(r) == \"-3/4\" and repr(r) == \"Ratio(-3, 4)\"\n    assert (R(10, 5).n, R(10, 5).d) == (2, 1) and (R(0, 7).n, R(0, 7).d) == (0, 1) and (R(-2, -6).n, R(-2, -6).d) == (1, 3)\n    try:\n        R(1, 0)\n        assert False, \"a denominator of 0 should raise ValueError\"\n    except ValueError:\n        pass\ndef test_conversions():\n    \"conversions\"\n    R = ns[\"Ratio\"]\n    assert float(R(3, 4)) == 0.75 and int(R(-3, 4)) == 0 and int(R(7, 2)) == 3 and int(R(-7, 2)) == -3\n    assert round(R(7, 4)) == 2 and round(R(1, 3), 2) == 0.33\n    assert bool(R(0, 3)) is False and bool(R(-1, 3)) is True\n    assert \"abc\"[R(4, 2)] == \"c\" and list(range(R(6, 3))) == [0, 1]\n    try:\n        [1, 2, 3][R(1, 2)]\n        assert False, \"a fraction can't be a list index\"\n    except TypeError:\n        pass\ndef test_format():\n    \"formatting\"\n    R = ns[\"Ratio\"]\n    assert [f\"{R(a, b):mixed}\" for a, b in [(7, 4), (-7, 4), (8, 4), (3, 4), (0, 5), (-1, 2)]] == [\"1 3/4\", \"-1 3/4\", \"2\", \"3/4\", \"0\", \"-1/2\"]\n    assert f\"{R(1, 3):.2f}\" == \"0.33\" and f\"{R(3, 4):.0%}\" == \"75%\" and f\"{R(2, 4)}\" == \"1/2\" and f\"{R(1, 8):>6}\" == \" 0.125\"")

## Warm-up

`len(full)` called `Shelf.__len__`. More surprising: `bool` and `if` used it too. The class never said what "true" means, so Python decided that a shelf with a length of 0 is false, just like an empty list. Python's built-in functions and syntax are connected to special methods like this throughout, and this week maps them.

## The data model

A **special method** (also called a *dunder* method, for the **d**ouble **under**scores) is one that Python calls for you when you use some piece of ordinary syntax or a built-in function. Together they're called Python's **data model**: the rules for how objects take part in the language. You've met many already:

| You write | Python calls | Met in |
|---|---|---|
| `Book(...)` | `__init__` | Week 28 |
| `repr(x)`, `str(x)`, `f"{x}"` | `__repr__`, `__str__`, `__format__` | Week 29 |
| `x == y`, `hash(x)` | `__eq__`, `__hash__` | Week 29 |
| `x + y`, `-x`, `abs(x)` | `__add__`, `__neg__`, `__abs__` | Week 29 |
| `bool(x)`, `if x:` | `__bool__`, else `__len__` | Week 29, today |
| `len(x)` | `__len__` | today |
| `int(x)`, `float(x)`, `round(x)` | `__int__`, `__float__`, `__round__` | today |
| `seq[x]`, `range(x)`, `hex(x)` with `x` as a whole number | `__index__` | today |
| `obj.missing` | `__getattr__` | today |
| `x[k]`, `x[k] = v`, `k in x`, `for v in x` | `__getitem__`, `__setitem__`, `__contains__`, `__iter__` | Day 2 |
| `x < y`, `sorted(xs)`, `max(xs)` | `__lt__` and friends | Day 3 |
| `x(...)` | `__call__` | Day 4 |

A few rules hold for all of them:

- **You define them; Python calls them.** Write `len(shelf)`, not `shelf.__len__()`. The built-in does extra work (checking the result, trying fallbacks), and code that calls special methods directly is harder to read.
- **Only use names Python defines.** Inventing your own `__shiny__` method doesn't make it special; it just looks as if it were.
- **They're looked up on the class, not the object.** Python goes straight to `type(x)` for special methods, so setting one on an instance has no effect:

This is why every special method in this course is written in the class body. (It also makes them fast: Python doesn't have to search each object first.)

## Size and truth

`len(x)` calls `x.__len__()`, and checks the answer: it must be a whole number of 0 or more, or `len` raises an error. Give a class `__len__` when it holds a number of things: a basket's items, a queue's people, a playlist's songs.

Truth uses size as a fallback. To decide `bool(x)` (and so `if x:`, `while x:`, `not x`, `and`, `or`), Python tries:

1. `__bool__`, which must return `True` or `False`;
2. if there's no `__bool__`, `__len__`: true if the length isn't 0;
3. if there's neither, the object is true.

So a container usually only needs `__len__`, and gets sensible truth for free. Write `__bool__` when truth means something other than "not empty", as for Week 29's vectors (`Vector(0, 0)` is false).

Both methods are strict about what they return:

`len` wants an `int`, so `self.pages // 2` (or `math.ceil`) is the fix. `__bool__` wants a real `bool`, so return `self.pence != 0` (or `bool(self.pence)`).

## Conversions

Some built-in functions turn an object into a value of another type, and each has a special method:

| Call | Method | Should return |
|---|---|---|
| `int(x)` | `__int__` | an `int` |
| `float(x)` | `__float__` | a `float` |
| `round(x)`, `round(x, n)` | `__round__(self, ndigits=None)` | usually an `int` for `round(x)`, the same kind as `x` for `round(x, n)` |
| `complex(x)` | `__complex__` | a `complex` |

Only write a conversion when there's one obvious answer. `int(Money(1249))` giving pence is reasonable; `int` of a `Book` would mean nothing, so a book shouldn't have `__int__`.

`__index__` is a stricter cousin of `__int__`. It says "this object **is** a whole number", so Python lets it be used wherever only an integer will do: list positions, slices, `range`, `bin`, `hex`. `__int__` alone isn't enough there, because `float` has `__int__` too, and `items[2.7]` should be an error, not a quiet rounding.

## Formatting

Week 29 met `__format__`: an f-string field like `f"{x:spec}"` calls `type(x).__format__(x, "spec")`, and `format(x, "spec")` does the same. The spec is just a string, and your class decides what it means. A common, tidy design is: a few specs of your own, an empty spec meaning `str(x)`, and anything else handed on to a value you hold, so all the usual number formats work too:

Unknown specs end up in `format(self.metres, spec)`, which raises `ValueError` for nonsense like `parsecs`, the same error Python's own types give.

## Forwarding with __getattr__

Day 5 of Week 30 built wrappers that hold an object and pass calls on to it. When the wrapped object has many methods, writing one forwarding method each is tedious. `__getattr__` solves that: Python calls it **only when normal lookup fails**, with the name that wasn't found.

- `shopping.append` isn't found on the `Logged` object or its class, so Python calls `__getattr__("append")`, which hands back the list's own method.
- `shopping.inner` and `shopping.used` **are** found normally, so `__getattr__` isn't called for them. That's what keeps it from running all the time.
- Special methods aren't forwarded: `len(shopping)` fails, because Python looks for `__len__` on the class, not through `__getattr__`. A wrapper must define those itself.

Two cautions. First, if `__getattr__` uses an attribute that might not exist yet (say, `self.inner` before `__init__` has set it), looking it up calls `__getattr__` again, and again: a `RecursionError`. Second, forwarding everything hides what a class can really do. Use it for genuine wrappers, and prefer plain methods otherwise. (There's also `__getattribute__`, called for **every** lookup; it's rarely needed and easy to get wrong.)

## Practice

## Quiz

## Daily challenge

## Before you go

Take one class you wrote in Weeks 28 to 30 and ask, for each row of today's first table, whether it would make sense for that class. Write down the ones that would. Tomorrow: container protocols, so your classes can use `[]`, `in` and `for` like lists and dictionaries.